# Chapter 15: More About Type Hints
*From: Fluent Python, 2nd Edition*

## TL;DR

- **`@overload`** lets you declare multiple type signatures for a single function so the checker infers precise return types per call pattern.
- **`TypedDict`** gives per-key type annotations to dicts used as records, but provides **zero runtime enforcement** (use pydantic for that).
- **`typing.cast`** is a no-op at runtime that tells the type checker to treat a value as a specific type -- use sparingly.
- **Generic classes** are created with `TypeVar` + `Generic[T]`; the big conceptual hurdle is **variance** (invariant, covariant, contravariant).
- **Generic static protocols** combine `Protocol[T_co]` with structural subtyping to define parameterized interfaces.

---
## 1. Overloaded Signatures with `@overload`

The `@typing.overload` decorator lets you declare **multiple type signatures** for one function. The type checker tries each overload in order and uses the first match to determine the return type. The actual implementation (without type hints) comes last.

This is critical when the return type **depends on the argument types**.

In [ ]:
from typing import overload, Union

# --- Overloaded signatures (for the type checker only) ---

@overload
def double(x: int) -> int: ...

@overload
def double(x: str) -> str: ...

# --- Actual implementation ---
def double(x):
    """Double a number or repeat a string."""
    if isinstance(x, int):
        return x * 2
    elif isinstance(x, str):
        return x * 2
    raise TypeError(f"Unsupported type: {type(x)}")

# Demonstrate
print(double(21))        # 42   -- type checker infers int
print(double("ha"))      # haha -- type checker infers str

Without `@overload`, the checker would see a single signature returning `Union[int, str]` for every call. With overloads, it knows `double(21)` returns `int` and `double("ha")` returns `str`.

**Key rules:**
- Overloaded signatures use `...` as the body (no implementation).
- The final function has the real body but **no type annotations**.
- Overloads are checked top-to-bottom; first match wins.

In [ ]:
# A more realistic example: typed sum with @overload
from typing import overload, TypeVar, Union
from collections.abc import Iterable
import functools, operator

T = TypeVar("T")
S = TypeVar("S")

@overload
def typed_sum(it: Iterable[T]) -> Union[T, int]: ...

@overload
def typed_sum(it: Iterable[T], /, start: S) -> Union[T, S]: ...

def typed_sum(it, /, start=0):
    return functools.reduce(operator.add, it, start)

# When start is omitted, return type includes int (the default 0)
result1 = typed_sum([1.5, 2.5, 3.0])
print(f"Sum without start: {result1}")

# When start is given, return type is Union[T, S]
result2 = typed_sum([1, 2, 3], 100)
print(f"Sum with start=100: {result2}")

---
## 2. TypedDict for Typed Dictionaries

`TypedDict` lets you annotate dicts used as records, giving each key its own value type. But it is **not** a data class builder -- at runtime it creates a plain `dict` with no enforcement.

In [ ]:
from typing import TypedDict

class BookDict(TypedDict):
    isbn: str
    title: str
    authors: list[str]
    pagecount: int

# Construction -- looks like a class, but produces a plain dict
book = BookDict(
    isbn="0134757599",
    title="Refactoring, 2e",
    authors=["Martin Fowler", "Kent Beck"],
    pagecount=478
)

print(type(book))           # <class 'dict'>
print(book["title"])        # Refactoring, 2e

# The annotations live on the TypedDict class, not on instances
print(BookDict.__annotations__)

**What the type checker enforces** (Mypy would flag these):
- Assigning wrong types to known keys
- Adding keys not in the TypedDict definition
- Deleting required keys

**What it does NOT enforce at runtime:**
- Nothing! `TypedDict` is a pure static-analysis construct.
- `json.loads()` returns `Any`, so assigning its result to a TypedDict variable gives no safety.

For runtime validation of JSON data, use **pydantic** instead.

In [ ]:
# Demonstrating that TypedDict has NO runtime protection
import json

# This invalid "book" is happily accepted at runtime
bad_json = '{"title": "Andromeda Strain", "flavor": "pistachio", "authors": true}'
not_a_book = json.loads(bad_json)
print(f"Type: {type(not_a_book)}")
print(f"Contents: {not_a_book}")
print(f"'authors' is a bool, not a list: {not_a_book['authors']}")
print()
print("TypedDict provides ZERO runtime checking.")
print("For runtime validation, use pydantic or manual isinstance checks.")

---
## 3. Type Casting with `typing.cast`

`typing.cast(typ, val)` returns `val` unchanged. It exists solely to tell the type checker: "trust me, this value is of type `typ`."

Use it to:
- Silence spurious type errors
- Work around incorrect or outdated type stubs

Avoid overusing it -- prefer `cast` over `# type: ignore` or `Any`, but treat frequent casts as a code smell.

In [ ]:
from typing import cast

# cast() is literally a no-op at runtime
def cast_demo(typ, val):
    """This is typing.cast's actual implementation."""
    return val

# Example: helping the checker after an isinstance guard
def find_first_str(items: list[object]) -> str:
    index = next(i for i, x in enumerate(items) if isinstance(x, str))
    # Without cast, checker sees a[index] as 'object', not 'str'
    return cast(str, items[index])

result = find_first_str([1, 2.0, "hello", None])
print(f"Found: {result!r} (type: {type(result).__name__})")

# Proving cast does nothing at runtime
x = cast(int, "I am a string")
print(f"cast(int, 'I am a string') = {x!r}  -- still a string!")

**Alternatives and trade-offs:**
| Approach | Pros | Cons |
|---|---|---|
| `cast(T, val)` | Explicit, documents intent | Still a lie if wrong |
| `# type: ignore` | Quick | Less informative |
| `Any` | Flexible | Contagious -- disables checking downstream |

Bottom line: an occasional `cast()` is fine. If you need many, rethink your type design.

---
## 4. Reading Type Hints at Runtime

Python stores type hints in `__annotations__` dicts. But you should use `typing.get_type_hints()` instead of reading `__annotations__` directly, because it resolves:
- Forward references (strings like `'Rectangle'`)
- Postponed annotations from `from __future__ import annotations`

In [ ]:
from typing import get_type_hints

def clip(text: str, max_len: int = 80) -> str:
    """Clip text to max_len characters."""
    end = 0
    if len(text) > max_len:
        space_before = text.rfind(" ", 0, max_len)
        end = space_before if space_before >= 0 else max_len
    else:
        end = len(text)
    return text[:end].rstrip()

# Raw annotations -- works, but may contain strings instead of types
print("Raw __annotations__:")
print(clip.__annotations__)
print()

# Preferred: get_type_hints resolves forward refs and strings
print("get_type_hints():")
print(get_type_hints(clip))

In [ ]:
# Forward reference problem: annotations as strings
# When a class method returns its own type, you must quote it
# (or use from __future__ import annotations)

class TreeNode:
    def __init__(self, value: int, children: "list[TreeNode] | None" = None):
        self.value = value
        self.children = children or []

    def add_child(self, value: int) -> "TreeNode":
        child = TreeNode(value)
        self.children.append(child)
        return child

# Raw annotations contain STRINGS, not types
print("Raw annotations:", TreeNode.__init__.__annotations__)
print()

# get_type_hints resolves the strings to actual types
resolved = get_type_hints(TreeNode.__init__)
print("Resolved hints:", resolved)

**Best practice:** Write a thin wrapper around `get_type_hints` so your code has a single point to update if the API changes:

```python
@classmethod
def _fields(cls) -> dict[str, type]:
    return get_type_hints(cls)
```

This pattern is used in the `Checked` class from Chapter 24.

---
## 5. Implementing Generic Classes

A generic class uses `TypeVar` and inherits from `Generic[T]`. Users parameterize it with actual types: `MyClass[int]`. The type checker then enforces consistency across all methods that use `T`.

In [ ]:
import random
from typing import TypeVar, Generic
from collections.abc import Iterable

T = TypeVar("T")

class LottoBlower(Generic[T]):
    """A generic lottery blower -- draws random items of type T."""

    def __init__(self, items: Iterable[T]) -> None:
        self._balls = list(items)

    def load(self, items: Iterable[T]) -> None:
        self._balls.extend(items)

    def pick(self) -> T:
        if not self._balls:
            raise LookupError("pick from empty LottoBlower")
        position = random.randrange(len(self._balls))
        return self._balls.pop(position)

    def loaded(self) -> bool:
        return bool(self._balls)

    def inspect(self) -> tuple[T, ...]:
        return tuple(self._balls)

# Usage: parameterize with int
random.seed(42)
machine = LottoBlower[int](range(1, 11))
print(f"Pick: {machine.pick()}")
print(f"Pick: {machine.pick()}")
print(f"Remaining: {machine.inspect()}")
print(f"Loaded? {machine.loaded()}")

**Jargon check:**
| Term | Example |
|---|---|
| **Generic type** | `LottoBlower[T]` |
| **Formal type parameter** | `T` in the class definition |
| **Parameterized type** | `LottoBlower[int]` |
| **Actual type parameter** | `int` in `LottoBlower[int]` |

---
## 6. Variance: Covariant, Contravariant, and Invariant

Variance describes how subtype relationships of **type parameters** affect subtype relationships of **generic types**. This is the most abstract concept in the chapter.

### The Cafeteria Analogy

A school cafeteria has rules about what dispensers and trash cans are acceptable:

| Concept | Rule | Direction |
|---|---|---|
| **Invariant** | Only a `Juice` dispenser -- not `Beverage`, not `OrangeJuice` | Exact match only |
| **Covariant** | A `Juice` dispenser OR any subtype (e.g., `OrangeJuice`) is OK | Same direction as subtyping |
| **Contravariant** | A `Biodegradable` trash can OR any supertype (e.g., `Refuse`) is OK | Opposite direction |

In [ ]:
from typing import TypeVar, Generic

# --- Type hierarchy ---
class Beverage:
    """Any beverage."""

class Juice(Beverage):
    """Any fruit juice."""

class OrangeJuice(Juice):
    """Delicious OJ."""

# --- INVARIANT dispenser (default TypeVar) ---
T = TypeVar("T")

class InvariantDispenser(Generic[T]):
    def __init__(self, beverage: T) -> None:
        self.beverage = beverage
    def dispense(self) -> T:
        return self.beverage

# With invariance, ONLY InvariantDispenser[Juice] is accepted.
# InvariantDispenser[OrangeJuice] would be REJECTED by Mypy!
# (At runtime Python doesn't enforce this, but the type checker does.)

juice_dispenser = InvariantDispenser(Juice())
print(f"Invariant dispenser holds: {type(juice_dispenser.beverage).__name__}")

# --- COVARIANT dispenser ---
T_co = TypeVar("T_co", covariant=True)

class CovariantDispenser(Generic[T_co]):
    def __init__(self, beverage: T_co) -> None:
        self.beverage = beverage
    def dispense(self) -> T_co:
        return self.beverage

# With covariance, both Juice and OrangeJuice dispensers are accepted.
oj_dispenser = CovariantDispenser(OrangeJuice())
print(f"Covariant dispenser holds: {type(oj_dispenser.beverage).__name__}")

In [ ]:
# --- CONTRAVARIANT trash can ---
from typing import TypeVar, Generic

class Refuse:
    """Any refuse."""

class Biodegradable(Refuse):
    """Biodegradable refuse."""

class Compostable(Biodegradable):
    """Compostable refuse."""

T_contra = TypeVar("T_contra", contravariant=True)

class TrashCan(Generic[T_contra]):
    """A trash can typed on what refuse it accepts."""
    def put(self, refuse: T_contra) -> None:
        print(f"Trashed: {type(refuse).__name__}")

# With contravariance, a TrashCan[Refuse] (more general) IS accepted
# where TrashCan[Biodegradable] is expected, because it can handle
# Biodegradable AND everything else.
# But TrashCan[Compostable] (too specific) would be REJECTED.

general_can: TrashCan[Refuse] = TrashCan()
general_can.put(Biodegradable())
general_can.put(Compostable())

print()
print("Variance rules of thumb:")
print("  - Output only (dispensers, iterators) -> covariant")
print("  - Input only (sinks, callbacks params) -> contravariant")
print("  - Both input AND output (mutable collections) -> invariant")

### Variance Rules of Thumb

1. **Output data** (return types, yielded values) -- **covariant** (`T_co`)
2. **Input data** (method parameters after construction) -- **contravariant** (`T_contra`)
3. **Both input and output** (mutable collections like `list`) -- **invariant** (plain `T`)
4. **When in doubt** -- use invariant (the safe default)

`Callable[[ParamType], ReturnType]` demonstrates rules 1+2: `ReturnType` is covariant, `ParamType` is contravariant.

---
## 7. Generic Static Protocols

A generic protocol combines `Protocol` with type parameters to define structural interfaces that are parameterized. The type parameter is typically **covariant** because protocols describe output behavior.

In [ ]:
import math
from typing import Protocol, TypeVar, runtime_checkable

T_co = TypeVar("T_co", covariant=True)

@runtime_checkable
class SupportsAbs(Protocol[T_co]):
    """Objects that support abs() returning type T_co."""
    def __abs__(self) -> T_co: ...

# --- A class that satisfies the protocol structurally ---
class Vector2d:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def __abs__(self) -> float:
        return math.hypot(self.x, self.y)

    def __repr__(self) -> str:
        return f"Vector2d({self.x}, {self.y})"

def is_unit(v: SupportsAbs[float]) -> bool:
    """True if the magnitude of v is close to 1."""
    return math.isclose(abs(v), 1.0)

# Test with our Vector2d
v0 = Vector2d(0, 1)
v1 = Vector2d(math.sqrt(2)/2, math.sqrt(2)/2)
v2 = Vector2d(1, 1)

print(f"{v0} is unit? {is_unit(v0)}")
print(f"{v1} is unit? {is_unit(v1)}")
print(f"{v2} is unit? {is_unit(v2)}")

# Also works with int and complex -- they implement __abs__
print(f"int 1 is unit? {is_unit(1)}")
print(f"complex(.5, sqrt(3)/2) is unit? {is_unit(complex(.5, math.sqrt(3)/2))}")

# Runtime protocol check
print(f"\nVector2d satisfies SupportsAbs? {isinstance(v0, SupportsAbs)}")

In [ ]:
# Another generic protocol: RandomPicker
from typing import Protocol, TypeVar, runtime_checkable
import random

T_co = TypeVar("T_co", covariant=True)

@runtime_checkable
class RandomPicker(Protocol[T_co]):
    """Any object with a pick() method returning T_co."""
    def pick(self) -> T_co: ...

# A simple implementation
class SimplePicker:
    def __init__(self, items: list[str]) -> None:
        self._items = list(items)

    def pick(self) -> str:
        return random.choice(self._items)

picker = SimplePicker(["apple", "banana", "cherry"])
print(f"Picked: {picker.pick()}")
print(f"Satisfies RandomPicker? {isinstance(picker, RandomPicker)}")

---
## Try It Yourself

In [ ]:
# Exercise 1: Write overloaded signatures for a function "stringify"
# that converts int -> str (via str()) and list -> str (via ", ".join())
# The type checker should know stringify(42) returns str and
# stringify(["a", "b"]) also returns str (but via different logic).

from typing import overload

@overload
def stringify(x: int) -> str: ...

@overload
def stringify(x: list[str]) -> str: ...

def stringify(x):
    if isinstance(x, int):
        return str(x)
    elif isinstance(x, list):
        return ", ".join(x)
    raise TypeError(f"Unsupported: {type(x)}")

# Test
print(stringify(42))             # Expected: "42"
print(stringify(["a", "b", "c"]))  # Expected: "a, b, c"

In [ ]:
# Exercise 2: Create a TypedDict for a configuration object with these keys:
#   host: str, port: int, debug: bool, tags: list[str]
# Then create a valid instance and print it.

from typing import TypedDict

class AppConfig(TypedDict):
    host: str
    port: int
    debug: bool
    tags: list[str]

config = AppConfig(
    host="localhost",
    port=8080,
    debug=True,
    tags=["web", "api"]
)

print(config)                    # Expected: plain dict
print(type(config))              # Expected: <class 'dict'>
print(config["port"])            # Expected: 8080

In [ ]:
# Exercise 3: Make a generic Stack class that is parameterized on element type.
# It should support push(item: T), pop() -> T, and is_empty() -> bool.

from typing import TypeVar, Generic

T = TypeVar("T")

class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def is_empty(self) -> bool:
        return len(self._items) == 0

    def __repr__(self) -> str:
        return f"Stack({self._items})"

# Test with int
int_stack = Stack[int]()
int_stack.push(1)
int_stack.push(2)
int_stack.push(3)
print(int_stack)                 # Expected: Stack([1, 2, 3])
print(int_stack.pop())           # Expected: 3
print(int_stack.is_empty())      # Expected: False

---
## Key Takeaways

1. **Use `@overload`** when the return type of a function depends on the types of its arguments -- declare each signature separately, implement once without annotations.
2. **`TypedDict` is for static analysis only** -- it annotates dict-as-record patterns but creates plain dicts at runtime with no validation.
3. **`typing.cast` is a runtime no-op** that guides the type checker; use it sparingly and prefer it over `# type: ignore` or `Any`.
4. **Read type hints via `get_type_hints()`**, not `__annotations__`, to handle forward references and postponed evaluation.
5. **Generic classes** use `TypeVar` + `Generic[T]` to create reusable, type-safe containers and abstractions.
6. **Variance matters for generic types**: covariant for outputs (read-only), contravariant for inputs (write-only), invariant for both (mutable containers).
7. **Generic protocols** (`Protocol[T_co]`) let you define structural interfaces parameterized on types, enabling precise type checking without inheritance.

## See Also

- [[overloaded-signatures]] -- Full details on `@overload` syntax and the `max` case study
- [[typeddict]] -- TypedDict patterns, limitations, and when to use pydantic instead
- [[type-casting]] -- When and how to use `typing.cast` safely
- [[runtime-type-hints]] -- Reading annotations at runtime and the PEP 563 situation
- [[generic-classes]] -- Implementing and using generic classes
- [[variance]] -- Deep dive into covariant, contravariant, and invariant types
- [[generic-static-protocols]] -- Building parameterized structural interfaces
- [Mypy documentation on generics](https://mypy.readthedocs.io/en/stable/generics.html)
- [PEP 484 -- Type Hints](https://peps.python.org/pep-0484/)
- [PEP 589 -- TypedDict](https://peps.python.org/pep-0589/)